# 06 — Tiny Qwen3: the full forward pass (M1.6)

A complete Qwen3-architecture causal language model, built only from our tested
modules (`qwen3_scratch.model.Qwen3ForCausalLM`). The weights are **random**: it is
an architecture check, not a trained language model. CPU only, no downloads. Plots
need `matplotlib`.

Write-up: `docs/components/model.md`.

## 1. Architecture overview

```
input_ids [B, T]
  → embed_tokens                [B, T, H]     a lookup table, V × H; no position embedding
  → N × decoder block           [B, T, H]     h += Attn(RMSNorm(h)); h += SwiGLU(RMSNorm(h))
      (RoPE tables computed once per forward, shared by all blocks)
  → final RMSNorm (model.norm)  [B, T, H]
  → lm_head (= embed_tokens.weight when tied)   [B, T, V]   logits
  → cross-entropy vs. labels shifted by one     scalar      loss
```

## 2–4. Configuration, initialisation, parameter count

In [ ]:
import math
from pathlib import Path

import torch

from qwen3_scratch.config import QWEN3_0_6B_BASE, Qwen3Config
from qwen3_scratch.generation import greedy_generate
from qwen3_scratch.loss import causal_lm_loss, shift_labels
from qwen3_scratch.model import Qwen3ForCausalLM

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

config_path = next(
    p for p in [Path("configs/qwen3_tiny.json"), Path("../configs/qwen3_tiny.json")] if p.exists()
)
cfg = Qwen3Config.from_json(config_path)
print(cfg)
_ = torch.manual_seed(0)
model = Qwen3ForCausalLM(cfg).eval()  # HF-style init: N(0, 0.02) for matrices, 1 for norms
print(
    f"\nparameters: {model.num_parameters():,} (config estimate {cfg.estimate_num_parameters():,})"
)

In [ ]:
groups = {}
for name, p in model.named_parameters():  # a tied matrix is listed once
    if "embed_tokens" in name:
        key = "embedding (= LM head, tied)"
    elif "norm.weight" in name and "layers" not in name:
        key = "final norm"
    else:
        part = name.split(".")[3]  # self_attn | mlp | input_layernorm | post_attention_layernorm
        key = {"self_attn": "attention (incl. QK-Norm)", "mlp": "SwiGLU MLP"}.get(
            part, "block RMSNorms"
        )
    groups[key] = groups.get(key, 0) + p.numel()
for k, v in groups.items():
    print(f"{k:28s} {v:>9,}  ({100 * v / model.num_parameters():5.1f} %)")

if plt is not None:
    fig, ax = plt.subplots(figsize=(6, 2.6))
    names, values = list(groups), list(groups.values())
    ax.barh(names, values, color="#2a78d6", height=0.6)
    for i, v in enumerate(values):
        ax.text(v, i, f" {v:,}", va="center", fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("parameters")
    ax.set_xlim(0, max(values) * 1.25)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_title("Tiny Qwen3 parameter budget")
    plt.show()

big = Qwen3ForCausalLM(QWEN3_0_6B_BASE, device="meta")  # shapes only, no memory
print(f"\nQwen3-0.6B architecture (meta device): {big.num_parameters():,} parameters")

## 5. Token embedding

`embed_tokens.weight` has one row of size $H$ per token id. The embedding is a pure
lookup: `embed_tokens(ids) == weight[ids]`. There is **no** absolute position
embedding: positions enter only through RoPE inside attention.

In [ ]:
ids = torch.tensor([[72, 101, 108, 108, 111]])  # "Hello" as UTF-8 bytes (toy tokenizer, V=256)
emb = model.model.embed_tokens(ids)
print("ids", tuple(ids.shape), "-> embeddings", tuple(emb.shape))
print("lookup == row indexing:", torch.equal(emb[0], model.model.embed_tokens.weight[ids[0]]))

## 6–8, 10. Stacked blocks, final RMSNorm, LM head: the full forward pass

In [ ]:
with torch.no_grad():
    out = model(ids, output_hidden_states=True)
stages = [("embeddings", out.hidden_states[0])]
stages += [(f"after block {i}", h) for i, h in enumerate(out.hidden_states[1:])]
stages += [("final RMSNorm", out.last_hidden_state), ("logits", out.logits)]
for name, t in stages:
    print(f"{name:16s} {tuple(t.shape)!s:16s} mean token-vector norm {t.norm(dim=-1).mean():8.3f}")

The residual stream (embeddings → block outputs) is never normalised inside the
blocks. The **final RMSNorm** brings it back to a fixed scale before the LM head.

## 9. Weight tying

With `tie_word_embeddings = true`, `lm_head.weight` **is** `embed_tokens.weight`:
the same `nn.Parameter` object, one storage and one gradient. Logits are dot
products of the final hidden state with every token embedding. That saves
$V\cdot H$ parameters (32,768 here; 155.6M for Qwen3-0.6B).

In [ ]:
print("same Parameter object:", model.lm_head.weight is model.model.embed_tokens.weight)
print(
    "same storage         :",
    model.lm_head.weight.data_ptr() == model.model.embed_tokens.weight.data_ptr(),
)
print(
    "logits == h @ Eᵀ     :",
    torch.equal(out.logits, out.last_hidden_state @ model.model.embed_tokens.weight.T),
)
untied = Qwen3ForCausalLM(Qwen3Config(**{**cfg.to_dict(), "tie_word_embeddings": False}))
extra = untied.num_parameters() - model.num_parameters()
print(f"untied model: {untied.num_parameters():,} parameters (+{extra:,})")

## 11. Next-token logits

In [ ]:
probs = torch.softmax(out.logits[0, -1], dim=-1)
top = torch.topk(probs, 8)
for p, i in zip(top.values.tolist(), top.indices.tolist(), strict=True):
    print(f"id {i:3d} {bytes([i]).decode('utf-8', 'replace')!r:6s} p = {p:.4f}")
print(f"uniform would be {1 / cfg.vocab_size:.4f}; max p = {probs.max():.4f}")

if plt is not None:
    fig, ax = plt.subplots(figsize=(6.5, 2.8))
    ax.bar(range(cfg.vocab_size), probs.numpy(), color="#2a78d6", width=1.0)
    ax.axhline(1 / cfg.vocab_size, color="#9a9a94", lw=1, ls="--")
    ax.text(
        cfg.vocab_size - 1,
        1 / cfg.vocab_size,
        " uniform 1/256",
        va="bottom",
        ha="right",
        fontsize=8,
    )
    ax.set_xlabel("token id")
    ax.set_ylabel("p(next token)")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_title("Untrained model: nearly uniform next-token distribution")
    plt.show()

## 12. Causal language-modelling loss

Teacher forcing: the whole true sequence is fed at once, and position $t$ is asked
to predict token $t+1$. The causal mask guarantees it has seen only tokens
$\le t$. With `labels = input_ids`, the shift is done **once**, inside the loss:

| position $t$ | reads tokens | target |
|---|---|---|
| 0 | $t_0$ | $t_1$ |
| 1 | $t_0..t_1$ | $t_2$ |
| … | … | … |
| T−2 | $t_0..t_{T-2}$ | $t_{T-1}$ |
| T−1 | everything | — (no next token) |

$\mathcal L = -\frac1N\sum \log \mathrm{softmax}(z_t)_{y_{t+1}}$. We pass **logits**:
`cross_entropy` fuses log-softmax and NLL as $z_y - \log\sum e^{z}$ with the max
subtracted, which is stable. Computing `log(softmax(z))` separately can underflow
to `log(0)`. Shifting twice (in the data *and* in the loss) silently trains the
model to predict two tokens ahead.

In [ ]:
labels = ids.clone()
print("labels         :", labels[0].tolist())
print("aligned targets:", shift_labels(labels)[0].tolist(), "(-100 = ignored)")
with torch.no_grad():
    loss = model(ids, labels=labels).loss
manual = -torch.log_softmax(out.logits[0, :-1].double(), -1)[range(4), labels[0, 1:]].mean()
print(
    f"loss {loss.item():.6f} | manual {manual.item():.6f} | ln(V) = {math.log(cfg.vocab_size):.6f}"
)
masked = labels.clone()
masked[0, :3] = -100  # e.g. prompt tokens we do not train on
print("loss on the last 2 targets only:", causal_lm_loss(out.logits, masked).item())

## 13. Backpropagation

In [ ]:
model.zero_grad()
batch = torch.randint(0, cfg.vocab_size, (4, 32), generator=torch.Generator().manual_seed(1))
model(batch, labels=batch).loss.backward()
for name in [
    "model.embed_tokens.weight",
    "model.layers.0.self_attn.q_proj.weight",
    "model.layers.0.self_attn.q_norm.weight",
    "model.layers.1.mlp.down_proj.weight",
    "model.norm.weight",
]:
    g = model.get_parameter(name).grad
    print(f"{name:42s} grad norm {g.norm():.4e}")
print(
    "every parameter received a gradient:",
    all(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters()),
)

### Causality check on the full model (mandatory test, reproduced)

In [ ]:
prefix = torch.tensor([[1, 2, 3, 4, 5]])
a = torch.cat([prefix, torch.tensor([[10, 11, 12]])], dim=1)
b = torch.cat([prefix, torch.tensor([[200, 201, 202]])], dim=1)
with torch.no_grad():
    la, lb = model(torch.cat([a, b])).logits
print("max |Δlogits| on shared prefix positions:", (la[:5] - lb[:5]).abs().max().item())
print("max |Δlogits| on later positions       :", (la[5:] - lb[5:]).abs().max().item())

## 14. Greedy generation (no KV cache)

In [ ]:
gen = greedy_generate(model, ids, max_new_tokens=10)
new = gen[0, ids.shape[1] :].tolist()
print("prompt :", ids[0].tolist(), repr(bytes(ids[0].tolist()).decode()))
print("new ids:", new, repr(bytes(new).decode("utf-8", "replace")))
print("deterministic:", torch.equal(gen, greedy_generate(model, ids, max_new_tokens=10)))
try:
    greedy_generate(model, torch.zeros(1, 120, dtype=torch.long), max_new_tokens=20)
except ValueError as err:
    print("context limit enforced:", err)

## 15. Numerical comparison with Hugging Face

In [ ]:
try:
    import transformers
    from transformers.models.qwen3 import modeling_qwen3 as hf_mod
except ImportError:
    print("transformers not installed -> HF comparison skipped (pip install -e '.[hf]')")
else:
    hf_cfg = transformers.Qwen3Config(
        vocab_size=cfg.vocab_size,
        hidden_size=cfg.hidden_size,
        intermediate_size=cfg.intermediate_size,
        num_hidden_layers=cfg.num_hidden_layers,
        num_attention_heads=cfg.num_attention_heads,
        num_key_value_heads=cfg.num_key_value_heads,
        head_dim=cfg.head_dim,
        max_position_embeddings=cfg.max_position_embeddings,
        rope_theta=cfg.rope_theta,
        rms_norm_eps=cfg.rms_norm_eps,
        tie_word_embeddings=cfg.tie_word_embeddings,
        attn_implementation="eager",
    )
    hf = hf_mod.Qwen3ForCausalLM(hf_cfg).eval()  # HF's own random init ...
    ours = Qwen3ForCausalLM(cfg).eval()
    ours.load_state_dict(hf.state_dict(), strict=True)  # ... copied into ours, all keys
    with torch.no_grad():
        r = hf(input_ids=batch, labels=batch)
        o = ours(batch, labels=batch)
    print("parameters:", ours.num_parameters(), "vs HF", sum(p.numel() for p in hf.parameters()))
    print(
        "HF tied:",
        hf.lm_head.weight is hf.model.embed_tokens.weight,
        "| ours tied:",
        ours.weights_tied,
    )
    print("logits bit-identical:", torch.equal(o.logits, r.logits))
    print("loss   bit-identical:", torch.equal(o.loss, r.loss), f"({o.loss.item():.6f})")

## 16. Why the generated text is meaningless

The weights are random draws from $\mathcal N(0, 0.02^2)$. Nothing connects token
ids to meaning yet, so the next-token distribution is nearly uniform, and the
loss is $\approx\ln V = 5.545$, the loss of guessing uniformly. The argmax is
decided by accidents of initialisation. With **tied** embeddings, an untrained
model often repeats the last token: the residual stream still mostly contains that
token's embedding, and the LM head scores tokens by similarity to the embedding
matrix. Real text needs either training (future milestone) or the official
pretrained weights (M2, loaded into this same architecture).

### Review questions
1. Where do positions enter the model, and why is there no position embedding?
2. Why must the loss shift be applied exactly once?
3. How many parameters does tying save for Qwen3-0.6B?
4. Why does greedy generation without a KV cache cost $O(T^2)$ forward work per new token?
5. Which tensor would you compare first if logits disagreed with Hugging Face?